# Preparation

In [ ]:
import json
import pandas as pd
from collections import Counter
import requests
import re
from pathlib import Path

In [ ]:
home = Path.home()

# Functions

In [ ]:
TOKEN_RE = re.compile(r"\d+|[^\W\d_]+|[.,/:;()\[\]-]")

def skeleton(text):
    out = []
    prev = None

    for tok in TOKEN_RE.findall(text):
        if tok[0].isdigit():
            kind = "N"
        elif tok[0].isalpha():
            kind = "W"
        else:
            kind = tok

        # Collapse consecutive words
        if kind == "W" and prev == "W":
            continue

        out.append(kind)
        prev = kind

    return "".join(out)

In [ ]:
def walk_keys(obj, prefix=""):
    if isinstance(obj, dict):
        for key, value in obj.items():
            path = f"{prefix}.{key}" if prefix else key
            yield path
            yield from walk_keys(value, path)

# Analysis

## Initial data

In [ ]:
structure_counter = Counter()


objects = []

with open(f"{home}/code/data/shbd/shb.jsonld.lines", "r", encoding="utf-8") as f:
	objects = [json.loads(row) for row in f]

skeleton_notes = []

example = {}
	
print(len(objects))

for object in objects:
	entity = object["@graph"][1]
	if "hasNote" in entity:
		note = entity["hasNote"][0]["label"]
		pattern = skeleton(note)
		skeleton_notes.append(pattern)
		structure_counter.update([pattern])

		example.setdefault(pattern, note)

print(len(skeleton_notes))
print(*skeleton_notes[:3], sep="\n")

### Count properties

In [ ]:
property_counts = Counter()
subject_counts = Counter()

for object in objects:
	entity = object["@graph"][1]
	property_counts.update(walk_keys(entity))
      
	subjects = entity.get("instanceOf", {}).get("subject", [])
    
	subject_counts.update(
          subject["@id"]
              for subject in subjects
                    )

In [ ]:
for key, count in property_counts.most_common():
    print(f"{count:>5}  {key}")

### Inspect structure of descriptions

In [ ]:
print("| Count | Pattern | Example |")
print("|------:|---------|---------|")

for pattern, count in structure_counter.most_common(20):
    ex = example[pattern].replace("|", "\\|")  # Escape pipes if any
    print(f"| {count} | `{pattern}` | {ex} |")

## Enriched data

In [ ]:
objects = []

with open(f"{home}/code/data/shbd/shb-cleaned-with-subjects.jsonld.lines", "r", encoding="utf-8") as f:
	objects = [json.loads(row) for row in f]
	
print(len(objects))


In [ ]:
property_counts = Counter()
subject_counts = Counter()

instances = []
for object in objects:
	entity = object["@graph"]["@graph"][1]
	property_counts.update(walk_keys(entity))
      
	subjects = entity.get("instanceOf", {}).get("subject", [])
	subject_counts.update(
          subject["@id"]
              for subject in subjects
                    )
	
	instances.append(entity)

print(instances[:3])

In [ ]:
extent = [{"@id": i["@id"], "extent": i["extent"][0]["label"]} for i in instances if "extent" in i and i["extent"][0]["label"][0].isalpha()]
extent_df = pd.json_normalize(extent)
extent_df.info()
extent_df.head(2)

pd.DataFrame(extent_df.value_counts(subset=["extent"])).head(10)


### Inspect seriesStatement

In [ ]:
series_membership = [{"@id": i["@id"], "seriesMembership": i["seriesMembership"][0]} for i in instances if "seriesMembership" in i]

print(series_membership[:3])    

In [ ]:
series_df = pd.json_normalize(series_membership)
series_df.info()
series_df.head(10)

In [ ]:
most_common_titles = pd.DataFrame(series_df.value_counts(subset=["seriesMembership.hasTitle.mainTitle"]))

most_common_titles.head(20)

In [ ]:
series_df[series_df["seriesMembership.hasTitle.mainTitle"] == "UNT 1959: julnr"]

### Count properties and subjects

#### Properties

In [ ]:
for key, count in property_counts.most_common():
    print(f"{count:>5}  {key}")

#### Subjects

In [ ]:
for key, count in subject_counts.most_common():

    print(f"{count:>5}  {key}")

# Match-resultat

In [ ]:
with_responsibility_statement = pd.read_json(f"{home}/code/data/shbd/results/match_id_map_20260814.json", orient="index")

with_responsibility_statement = pd.json_normalize(with_responsibility_statement["best_match"])

with_responsibility_statement

In [ ]:
with_fuzzy_free_text = pd.read_json(f"{home}/code/data/shbd/results/match_id_map_20260817.json", orient="index")

with_fuzzy_free_text = pd.json_normalize(with_fuzzy_free_text["best_match"])

with_fuzzy_free_text

In [ ]:
len(with_responsibility_statement[with_responsibility_statement["score"] < 1.5])

In [ ]:
len(with_fuzzy_free_text[with_fuzzy_free_text["score"] < 1.5])

# Random stuff

In [ ]:
headers = {"Accept": "application/ld+json"}

query_string = f"instanceType:PhysicalResource title:({shbd_prepepd['mainTitle']}) title:({shbd_prepepd['subtitle']}) contributor:{shbd_prepepd['responsibility_statement']}* {shbd_prepepd['part_of_issn']} {shbd_prepepd['issn_from_note']}"


params = {"_q": "title:Hembergska+huset+i+Simrishamn contributor:Ehrnberg, G.*",
          #"_embellished": "false", Den här verkar inte göra något
          "_lens": "chips", # Den här behöver vara i plural
          "_stats": "false",
          "limit": 10}

res = requests.get("http://libris.kb.se/find?", params = params, headers=headers)
res.raise_for_status()
print(res.url)

print("Status:", res.status_code)
print("Number of results:", res.json()["totalItems"])
print("\nResult keys:", *res.json().keys(), sep=", ")

# Var finns den vanliga bibliografiska datan?
records = res.json()["items"]
print("\nItem keys:", *records[0].keys(), sep=", ")

print(records[0])


In [ ]:
print(res.json()["stats"])

In [ ]:
import re
rest = "Swedenborg : sökaren i naturens och andens värld :hans verk och efterföljd / Carl / Hej"
#rest = "Egerbladh, Ossian, Ur Lappmarkens bebyggelsehistoria. Umeå. 1-8. Se SHB 1961/70:7809.9 : Barsele : minnesskrift med anledning av byns tvåhundraåriga tillvaro.1970. 98 s.10 : Stensele 1741-1860 : de hundra äldsta nybyggesupptagningarna.1972. 237 s.11 : Fyra gamla Lyckselebyar : Björksele, Brattfors, Falträsk, Vägsele :denna utredning har utförts med anledning av Lycksele sockens 300-årsjubileum. 1973. 94 s. : ill."
subtitle = ""
title= ""
if ' : ' in rest:
	title, rest = rest.split(' : ', 1)
	print (title)
	print(rest)

	if ':' in rest:
		parts =  re.split(r" ([./])", rest, maxsplit=1)
		subtitle = parts[0]
		if len(parts) > 1:
			print(parts)
			rest = "".join(parts[1:])
print()
print(title)
print(subtitle)
print(rest)



In [ ]:
csv_file = f"{home}/code/libris/repositories/librisxl/whelktool/scripts/dataimports/shb/data/mönster per materialtyp_rows.tsv"
patterns = pd.read_csv(csv_file, sep="\t", header=None, names=["year_range", "type", "pattern"])
patterns["pattern"] = patterns["pattern"].str.strip().str.replace(r"\s+", " ", regex=True)

patterns.info()
patterns.head(2)

In [ ]:
SYNTAX_ERAS = {
    "1771-1874": "early", 
    "1875-1900": "early",
    "1901-1920": "early",
    "1921-1935": "parenthesized",  # Serietillhörighet och källpublikation anges inom parentes. Sidor anges efter Ort/år
    "1936-1950": "parenthesized",  # -||- . -||-
    "1951-1960": "parenthesized", # Serietillhörighet och källpublikation anges inom parentes. Sidor anges före Ort/år
    "1961-1970": "dash_style", # Serietillhörighet och källpublikation anges efter ". -". Sidor anges efter Ort/år
    "1971-1975": "isbd_transition", # Kolon ":" mellan huvudtitel och undertitel. Serietillhörighet anges inom parentes efter ". -". Bidrag: Källpublikation anges efter ". - I: "
    "1976": "isbd", # -||- ". - " anges före nytt avsnitt (utgivning, omfång, serietillhörighet). -||- . -||- . ISSN anges
}

patterns["syntax_era"] = patterns["year_range"].map(SYNTAX_ERAS)

In [ ]:
unique_patterns = patterns.groupby(["pattern", "syntax_era"], as_index = False).agg({'year_range': ', '.join, 'type': 'first', 'syntax_era': 'first'})
unique_patterns["case"] = unique_patterns["year_range"] + " " + unique_patterns["type"]

unique_patterns.info()
unique_patterns.head(50)



In [ ]:

test_cases = ""

for record in unique_patterns.to_dict(orient="records"):
    
    
    test_cases += f"""
def test_parse_note_{record["case"].replace(" ", "_").replace("-", "_").replace("(", "").replace(")", "").replace(":", "").replace(",", "")}():
	instance = {{"hasNote": [{{"label": "{record["pattern"]}"}}]}}
	
	result = parse_note(instance, "{record["syntax_era"]}")

	assert result["category"] == None
	assert result["hasTitle"]["mainTitle"] == None
	assert result["hasTitle"]["subtitle"] == None
	assert result["responsibilityStatement"] == None
	assert result["extent"] == None
	assert result["partOf"] == None
	assert result["seriesMembership"] == None
	assert result["hasNote"] == None
		"""

print(test_cases)

In [ ]:
import time

start = time.time()

request_url = "https://libris.kb.se/find?_q=titel%3A%28ett+system+s%C3%A5+magnifikt+att+det+bl%C3%A4ndar%29+responsibilityStatement%3A%28amanda+svensson%29&_limit=20"

for r in range(20000):
	res = requests.get(request_url)
	if r % 500 == 0:
		elapsed = r / (time.time() - start)
		elapsed_formatted = "{0:.4g}".format(elapsed)
		print(
			f"{r + 1} records processed\t\t{elapsed_formatted} records/sec."
		)
	time.sleep(0.01)
